In [5]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import statistics

# === Configuration ===
PINECONE_API_KEY = "pcsk_4DqA35_BYaWgQCoVUkGTNDRYenf3NQzwzZX6C685nC2fwMj5qXgnpMXmUcH1eVXRjVfMgw"
PINECONE_INDEX_NAME = "quickstart-new-test-pdf-rows-final"
PINECONE_REGION = "us-west-2"  # use correct region if different

# === Initialize Pinecone (new SDK) ===
pc = Pinecone(api_key=PINECONE_API_KEY)

# === Connect to index (assumes it already exists, else create) ===
index = pc.Index(
    name=PINECONE_INDEX_NAME,
    dimension=384,  # Set to your model's dimension
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region=PINECONE_REGION)
)

# === Load embedding model ===
model = SentenceTransformer("all-MiniLM-L6-v2")

# === Sample Patient Data ===
age = 56
bpaarm = 78.0
bpacsz = 1.0
bpxpls = 56.0
bpxpuls = 170.0
bpxsy = [132.0, 134.0, 136.0, 138.0]
bpxdi = [72.0, 68.0, 70.0, 46.0]

# === Generate Questions ===
questions = [
    f"What is the clinical significance of an arm circumference of {bpaarm:.0f} cm when measuring blood pressure?",
    f"What blood pressure cuff size is recommended for a person with an arm circumference of {bpaarm:.0f} cm and cuff code {bpacsz:.0f}?",
    f"Is a pulse rate of {bpxpls:.0f} bpm considered bradycardia in adults?",
    f"What does a pulse of {bpxpuls:.0f} bpm indicate in a {int(age)}-year-old adult?",
    f"Is a systolic pressure of {bpxsy[0]:.0f} and diastolic of {bpxdi[0]:.0f} considered stage 1 hypertension?",
    f"Does a blood pressure reading of {bpxsy[3]:.0f}/{bpxdi[3]:.0f} suggest hypotension for a {int(age)}-year-old?",
    "What are the risks associated with large discrepancies between multiple blood pressure readings?",
    f"What is the clinical interpretation of an average blood pressure of {statistics.mean(bpxsy):.0f}/{statistics.mean(bpxdi):.0f} mmHg?",
    f"What are common symptoms or warning signs of diabetes in adults aged {int(age)}?",
    f"How does high blood pressure (e.g., {bpxsy[0]:.0f}/{bpxdi[0]:.0f}) correlate with the risk of developing type 2 diabetes in someone aged {int(age)}?"
]

# === Select a specific question by index (0-9), or set to None to run all ===
selected_query = 1  # Change to None to run all questions

print("\nTop Medical Questions and Pinecone Search Results:\n")

# Choose questions based on selection
if selected_query is not None and 0 <= selected_query < len(questions):
    questions_to_run = [questions[selected_query]]
else:
    questions_to_run = questions  # Run all if no specific selection

# Query loop
for idx, query in enumerate(questions_to_run):
    # Print the actual query being run
    if selected_query is not None:
        print(f"Selected Query [{selected_query}]: {query}\n")
    else:
        print(f"Query [{idx}]: {query}\n")
    
    query_vector = model.encode(query).tolist()
    results = index.query(
        vector=query_vector,
        top_k=5,
        include_metadata=True
    )

    for match in results.get("matches", []):
        score = match.get("score", 0)
        metadata = match.get("metadata", {})
        print(f"- Score: {score:.4f}")
        print(f"  File: {metadata.get('file', 'N/A')}")
        print(f"  Page: {metadata.get('page', 'N/A')}")
        print(f"  Text: {metadata.get('text', 'No text available')}\n")
    print("-" * 60)



Top Medical Questions and Pinecone Search Results:

Selected Query [1]: What blood pressure cuff size is recommended for a person with an arm circumference of 78 cm and cuff code 1?

- Score: 0.2876
  File: bpx_i_cleaned.pdf
  Page: 207.0
  Text: bpx_i_cleaned.pdf | Page 207: 92373.00 |  |  | 1.00 | 4.00 | 70.00 | 1.00 | 1.00 | 180.00 |  |  |  | 136.00 | 72.00 | 2.00 | 132.00 | 74.00 | 2.00 | 150.00 | 66.00 | 2.00

- Score: 0.2790
  File: bpx_i_cleaned.pdf
  Page: 151.0
  Text: bpx_i_cleaned.pdf | Page 151: 90027.00 |  |  | 1.00 | 5.00 | 62.00 | 1.00 | 1.00 | 180.00 | 166.00 | 102.00 | 2.00 | 172.00 | 100.00 | 2.00 | 168.00 | 102.00 | 2.00 |  |  | 

- Score: 0.2786
  File: bpx_i_cleaned.pdf
  Page: 181.0
  Text: bpx_i_cleaned.pdf | Page 181: 91272.00 |  |  | 1.00 | 4.00 | 70.00 | 1.00 | 1.00 | 180.00 | 170.00 | 66.00 | 2.00 | 172.00 | 44.00 | 2.00 | 164.00 | 0.00 | 2.00 |  |  | 

- Score: 0.2784
  File: bpx_i_cleaned.pdf
  Page: 184.0
  Text: bpx_i_cleaned.pdf | Page 184: 91416.00 |  